# Predicting AQI - v3 (Feature Engineering Focus)

Platform I/O:
- Input: `dataset/public/train.csv`, `dataset/public/test.csv`, `dataset/public/sample_submission.csv`
- Output: `working/submission.csv`

v3 focus:
- Rich feature engineering (temporal, meteorological, geospatial, group-relative)
- Leakage-safe fold target statistics
- Time-aware expanding validation
- Ensemble + OOF weight search


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer

from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def locate_data_dir():
    candidates = [
        'dataset/public',
        './dataset/public',
        '/kaggle/input/predicting-air-quality-index',
        '/kaggle/input/predicting-air-quality-index-dataset',
        '/Users/songling/Desktop/Predicting Air Quality Index',
    ]
    for d in candidates:
        if all(os.path.exists(os.path.join(d, f)) for f in ['train.csv', 'test.csv', 'sample_submission.csv']):
            return d
    raise FileNotFoundError('Cannot locate dataset files.')


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


In [ ]:
DATA_DIR = locate_data_dir()
print('Using DATA_DIR:', DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('train:', train.shape, 'test:', test.shape)


In [ ]:
# -----------------------------
# Feature Engineering
# -----------------------------

def build_base(df):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['ts'] = pd.to_datetime(dict(year=out['year'], month=out['month'], day=out['day'])) + pd.to_timedelta(out['hour'], unit='h')

    # temporal basics
    out['date_ordinal'] = out['date'].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
    out['dayofyear'] = out['date'].dt.dayofyear.astype(float)
    out['weekofyear'] = out['date'].dt.isocalendar().week.astype(float)
    out['quarter'] = out['date'].dt.quarter.astype(float)
    out['is_month_start'] = out['day'].isin([1, 2, 3]).astype(int)
    out['is_month_end'] = out['day'].isin([28, 29, 30, 31]).astype(int)

    # cyclical
    dow_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
    out['dow_num'] = out['day_of_week'].map(dow_map).fillna(0).astype(float)
    out['hour_sin'] = np.sin(2*np.pi*out['hour']/24.0)
    out['hour_cos'] = np.cos(2*np.pi*out['hour']/24.0)
    out['month_sin'] = np.sin(2*np.pi*out['month']/12.0)
    out['month_cos'] = np.cos(2*np.pi*out['month']/12.0)
    out['dow_sin'] = np.sin(2*np.pi*out['dow_num']/7.0)
    out['dow_cos'] = np.cos(2*np.pi*out['dow_num']/7.0)
    out['doy_sin'] = np.sin(2*np.pi*out['dayofyear']/365.25)
    out['doy_cos'] = np.cos(2*np.pi*out['dayofyear']/365.25)

    # meteorological nonlinear
    out['temp2'] = out['temperature'] ** 2
    out['hum2'] = out['humidity'] ** 2
    out['wind2'] = out['wind_speed'] ** 2
    out['vis2'] = out['visibility'] ** 2

    out['temp_humidity'] = out['temperature'] * out['humidity']
    out['temp_wind'] = out['temperature'] * out['wind_speed']
    out['hum_wind'] = out['humidity'] * out['wind_speed']
    out['wind_vis_ratio'] = out['wind_speed'] / (out['visibility'] + 1e-3)
    out['temp_hum_ratio'] = out['temperature'] / (out['humidity'] + 1e-3)
    out['visibility_inv'] = 1.0 / (out['visibility'] + 1e-3)

    # pseudo dew point (rough approximation)
    out['dew_point_like'] = out['temperature'] - ((100.0 - out['humidity']) / 5.0)
    out['temp_minus_dew'] = out['temperature'] - out['dew_point_like']

    # geospatial
    out['lat_lon_sum'] = out['latitude'] + out['longitude']
    out['lat_lon_diff'] = out['latitude'] - out['longitude']
    out['lat_x_lon'] = out['latitude'] * out['longitude']
    out['lat2'] = out['latitude'] ** 2
    out['lon2'] = out['longitude'] ** 2

    # bins / regimes (as category-like strings)
    out['hour_bin'] = pd.cut(out['hour'], bins=[-1,5,11,17,23], labels=['night','morning','afternoon','evening']).astype(str)
    out['temp_bin'] = pd.cut(out['temperature'], bins=[-np.inf,12,20,28,35,np.inf], labels=['very_cold','cool','mild','warm','hot']).astype(str)
    out['hum_bin'] = pd.cut(out['humidity'], bins=[-np.inf,35,55,75,90,np.inf], labels=['dry','normal','humid','wet','very_wet']).astype(str)
    out['vis_bin'] = pd.cut(out['visibility'], bins=[-np.inf,1,2,4,6,np.inf], labels=['very_low','low','mid','high','very_high']).astype(str)

    # crosses
    out['city_station'] = out['city'].astype(str) + '_' + out['station'].astype(str)
    out['station_hour'] = out['station'].astype(str) + '_' + out['hour'].astype(str)
    out['station_month'] = out['station'].astype(str) + '_' + out['month'].astype(str)
    out['city_hour'] = out['city'].astype(str) + '_' + out['hour'].astype(str)
    out['city_season'] = out['city'].astype(str) + '_' + out['season'].astype(str)
    out['station_dow'] = out['station'].astype(str) + '_' + out['day_of_week'].astype(str)
    out['city_month'] = out['city'].astype(str) + '_' + out['month'].astype(str)
    out['station_hour_bin'] = out['station'].astype(str) + '_' + out['hour_bin'].astype(str)

    return out


def add_train_based_group_feature_stats(train_df, apply_df):
    tr = train_df.copy()
    ap = apply_df.copy()

    weather_cols = ['temperature', 'humidity', 'wind_speed', 'visibility']

    # group-relative weather deviations
    group_keys = ['station', 'city', 'station_hour', 'city_hour', 'station_month']
    for g in group_keys:
        agg = tr.groupby(g)[weather_cols].mean()
        for c in weather_cols:
            ap[f'{c}_mean_by_{g}'] = ap[g].map(agg[c])
            ap[f'{c}_dev_by_{g}'] = ap[c] - ap[f'{c}_mean_by_{g}']

    # city centroid and distance-like features
    city_geo = tr.groupby('city')[['latitude','longitude']].mean()
    ap['city_lat_center'] = ap['city'].map(city_geo['latitude'])
    ap['city_lon_center'] = ap['city'].map(city_geo['longitude'])
    ap['dist_city_center_l1'] = (ap['latitude'] - ap['city_lat_center']).abs() + (ap['longitude'] - ap['city_lon_center']).abs()
    ap['dist_city_center_l2'] = np.sqrt((ap['latitude'] - ap['city_lat_center'])**2 + (ap['longitude'] - ap['city_lon_center'])**2)

    # station static meteorology priors from train X only
    st_weather = tr.groupby('station')[weather_cols].mean()
    for c in weather_cols:
        ap[f'station_mean_{c}'] = ap['station'].map(st_weather[c])
        ap[f'{c}_minus_station_mean'] = ap[c] - ap[f'station_mean_{c}']

    return ap


def smooth_target_stats(ref_x, ref_y, apply_x, key, m=120):
    gm = float(ref_y.mean())
    tmp = ref_x[[key]].copy()
    tmp['_y'] = ref_y.values
    agg = tmp.groupby(key)['_y'].agg(['mean', 'count', 'median', 'std'])

    mean_sm = (agg['mean'] * agg['count'] + gm * m) / (agg['count'] + m)

    out_mean = apply_x[key].map(mean_sm).fillna(gm)
    out_cnt = np.log1p(apply_x[key].map(agg['count']).fillna(0.0))
    out_median = apply_x[key].map(agg['median']).fillna(gm)
    out_std = apply_x[key].map(agg['std']).fillna(0.0)
    return out_mean, out_cnt, out_median, out_std


def add_fold_target_features(ref_x, ref_y, apply_x):
    ap = apply_x.copy()
    keys = [
        'station','city','season','day_of_week','hour','month',
        'city_station','station_hour','station_month','city_hour','city_season',
        'station_dow','city_month','hour_bin','temp_bin','hum_bin','vis_bin','station_hour_bin'
    ]
    for k in keys:
        m, c, med, s = smooth_target_stats(ref_x, ref_y, ap, k, m=140)
        ap[f'te_mean_{k}'] = m
        ap[f'te_cnt_{k}'] = c
        ap[f'te_median_{k}'] = med
        ap[f'te_std_{k}'] = s

    # date bucket target trend
    gm = float(ref_y.mean())
    b_ref = (ref_x['date_ordinal'] // 7).astype('int64')
    b_ap = (ap['date_ordinal'] // 7).astype('int64')
    btmp = pd.DataFrame({'b': b_ref.values, 'y': ref_y.values})
    bmean = btmp.groupby('b')['y'].mean()
    ap['te_week_bucket_mean'] = b_ap.map(bmean).fillna(gm)

    return ap


In [ ]:
train_fe = build_base(train)
test_fe = build_base(test)

# add train-X-based (non-target) stats features
train_fe = add_train_based_group_feature_stats(train_fe, train_fe)
test_fe = add_train_based_group_feature_stats(train_fe, test_fe)

# target and ids
y = train_fe['aqi'].astype(float)
train_ids = train_fe['id'].astype(int).values
test_ids = test_fe['id'].astype(int).values

X_all = train_fe.drop(columns=['aqi']).copy()
X_test = test_fe.copy()

# preserve time column for splitting only
time_values = X_all['ts'].copy()

X_all = X_all.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

# expanding time splits
order = np.argsort(time_values.values)
parts = np.array_split(order, 7)  # 6 folds
splits = []
for i in range(1, len(parts)):
    tr_idx = np.concatenate(parts[:i])
    va_idx = parts[i]
    if len(tr_idx) > 0 and len(va_idx) > 0:
        splits.append((tr_idx, va_idx))

print('Folds:', len(splits))
for i, (tr_idx, va_idx) in enumerate(splits, 1):
    print(f'Fold {i}: train={len(tr_idx)} valid={len(va_idx)}')


In [ ]:
# -----------------------------
# Models
# -----------------------------

cat_cols = [
    'day_of_week','season','city','station','city_station','station_hour','station_month','city_hour','city_season',
    'station_dow','city_month','hour_bin','temp_bin','hum_bin','vis_bin','station_hour_bin'
]

# drop pure time column from modeling
base_cols = [c for c in X_all.columns if c != 'ts']
num_cols = [c for c in base_cols if c not in cat_cols]

lgb_params_list = [
    dict(name='lgb_raw_a', objective='regression', metric='l2', n_estimators=4200, learning_rate=0.011,
         num_leaves=256, min_child_samples=14, subsample=0.92, colsample_bytree=0.92,
         reg_alpha=0.06, reg_lambda=1.25, random_state=42, n_jobs=-1),
    dict(name='lgb_raw_b', objective='regression', metric='l2', n_estimators=3600, learning_rate=0.013,
         num_leaves=192, min_child_samples=18, subsample=0.90, colsample_bytree=0.90,
         reg_alpha=0.10, reg_lambda=1.40, random_state=2024, n_jobs=-1),
    dict(name='lgb_log', objective='regression', metric='l2', n_estimators=4200, learning_rate=0.011,
         num_leaves=224, min_child_samples=16, subsample=0.92, colsample_bytree=0.92,
         reg_alpha=0.08, reg_lambda=1.30, random_state=3407, n_jobs=-1),
]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', make_ohe())]), cat_cols)
], remainder='drop')

etr_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', ExtraTreesRegressor(
            n_estimators=850,
            min_samples_leaf=2,
            max_features=0.78,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)

hgb_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', HistGradientBoostingRegressor(
            learning_rate=0.022,
            max_depth=12,
            max_iter=1300,
            min_samples_leaf=18,
            l2_regularization=0.20,
            random_state=RANDOM_STATE
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)


In [ ]:
# -----------------------------
# OOF predictions
# -----------------------------
model_oof = {}
model_test = {}

# helper prior model
def prior_predict(ref_x, ref_y, q_x):
    gm = float(ref_y.mean())
    ref = ref_x.copy()
    ref['y'] = ref_y.values

    def get(key):
        g = ref.groupby(key)['y'].agg(['mean','count'])
        p = q_x[key].map(g['mean']).fillna(gm)
        c = q_x[key].map(g['count']).fillna(0.0)
        return p, c

    p1, c1 = get('station_hour')
    p2, c2 = get('station')
    p3, c3 = get('city_hour')
    p4, c4 = get('city')
    p5, c5 = get('station_dow')

    w1 = np.log1p(c1)
    w2 = np.log1p(c2)
    w3 = np.log1p(c3)
    w4 = np.log1p(c4)
    w5 = np.log1p(c5)
    ws = w1 + w2 + w3 + w4 + w5 + 1e-9
    pred = (w1*p1 + w2*p2 + w3*p3 + w4*p4 + w5*p5) / ws
    return pred.values

for cfg in lgb_params_list:
    name = cfg['name']
    params = cfg.copy()
    params.pop('name')

    oof = np.zeros(len(X_all), dtype=float)
    te_folds = []

    for tr_idx, va_idx in splits:
        Xtr0 = X_all.iloc[tr_idx].copy()
        ytr = y.iloc[tr_idx].copy()
        Xva0 = X_all.iloc[va_idx].copy()
        Xte0 = X_test.copy()

        # fold-safe target features
        Xtr = add_fold_target_features(Xtr0.drop(columns=['ts']), ytr, Xtr0.drop(columns=['ts']))
        Xva = add_fold_target_features(Xtr0.drop(columns=['ts']), ytr, Xva0.drop(columns=['ts']))
        Xte = add_fold_target_features(Xtr0.drop(columns=['ts']), ytr, Xte0.drop(columns=['ts']))

        for c in cat_cols:
            Xtr[c] = Xtr[c].astype('category')
            Xva[c] = Xva[c].astype('category')
            Xte[c] = Xte[c].astype('category')

        model = LGBMRegressor(**params)

        if name == 'lgb_log':
            model.fit(
                Xtr, np.log1p(ytr),
                eval_set=[(Xva, np.log1p(y.iloc[va_idx]))],
                eval_metric='l2',
                callbacks=[early_stopping(200, verbose=False), log_evaluation(0)]
            )
            oof[va_idx] = np.expm1(model.predict(Xva, num_iteration=model.best_iteration_))
            te_folds.append(np.expm1(model.predict(Xte, num_iteration=model.best_iteration_)))
        else:
            model.fit(
                Xtr, ytr,
                eval_set=[(Xva, y.iloc[va_idx])],
                eval_metric='l2',
                callbacks=[early_stopping(200, verbose=False), log_evaluation(0)]
            )
            oof[va_idx] = model.predict(Xva, num_iteration=model.best_iteration_)
            te_folds.append(model.predict(Xte, num_iteration=model.best_iteration_))

    model_oof[name] = oof
    model_test[name] = np.mean(np.vstack(te_folds), axis=0)

# Extra models
for name, template in [('etr', etr_template), ('hgb', hgb_template)]:
    oof = np.zeros(len(X_all), dtype=float)
    te_folds = []
    for tr_idx, va_idx in splits:
        Xtr = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
        ytr = y.iloc[tr_idx].copy()
        Xva = X_all.iloc[va_idx].drop(columns=['ts']).copy()
        Xte = X_test.drop(columns=['ts']).copy()

        mdl = template
        mdl.fit(Xtr, ytr)
        oof[va_idx] = mdl.predict(Xva)
        te_folds.append(mdl.predict(Xte))

    model_oof[name] = oof
    model_test[name] = np.mean(np.vstack(te_folds), axis=0)

# Prior
oof_prior = np.zeros(len(X_all), dtype=float)
te_prior_folds = []
for tr_idx, va_idx in splits:
    Xtr = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
    ytr = y.iloc[tr_idx].copy()
    Xva = X_all.iloc[va_idx].drop(columns=['ts']).copy()
    Xte = X_test.drop(columns=['ts']).copy()

    oof_prior[va_idx] = prior_predict(Xtr, ytr, Xva)
    te_prior_folds.append(prior_predict(Xtr, ytr, Xte))

model_oof['prior'] = oof_prior
model_test['prior'] = np.mean(np.vstack(te_prior_folds), axis=0)

for k, p in model_oof.items():
    m = p != 0
    print(f'{k} OOF MSE:', round(mean_squared_error(y[m], p[m]), 6))


In [ ]:
# -----------------------------
# Blend + Post-process
# -----------------------------
names = list(model_oof.keys())
P = np.column_stack([model_oof[n] for n in names])
Pt = np.column_stack([model_test[n] for n in names])

valid_mask = np.any(P != 0, axis=1)
Pv = P[valid_mask]
yv = y.values[valid_mask]
station_v = train['station'].iloc[np.where(valid_mask)[0]].astype(str).values

# weighted objective for tail
q70 = np.quantile(yv, 0.70)
w = np.where(yv >= q70, 1.8, 1.0)

def wmse(y_true, y_pred, wgt):
    return float(np.sum(wgt * (y_true - y_pred)**2) / np.sum(wgt))

best = (1e18, None)
rng = np.random.default_rng(RANDOM_STATE)
for _ in range(150000):
    ww = rng.dirichlet(np.ones(len(names))*1.2)
    pred = Pv @ ww
    s = wmse(yv, pred, w)
    if s < best[0]:
        best = (s, ww)

best_w = best[1]
print('Best weighted score:', best[0])
print('Weights:', {n: float(v) for n, v in zip(names, best_w)})

oof_blend = Pv @ best_w
test_blend = Pt @ best_w

# station calibration
cal_df = pd.DataFrame({'station': station_v, 'pred': oof_blend, 'y': yv})
global_bias = float(np.mean(yv - oof_blend))
cal = {}
for st, g in cal_df.groupby('station'):
    x = g['pred'].values
    t = g['y'].values
    n = len(g)
    xm = x.mean()
    tm = t.mean()
    var = np.mean((x - xm)**2)
    if var < 1e-9:
        a = 1.0
    else:
        a = np.mean((x-xm)*(t-tm)) / (var + 1e-9)
    b = tm - a*xm
    lam = n / (n + 80.0)
    a = lam*a + (1-lam)*1.0
    b = lam*b + (1-lam)*global_bias
    cal[st] = (a, b)

pred = np.empty_like(test_blend)
for i, st in enumerate(test['station'].astype(str).values):
    a, b = cal.get(st, (1.0, global_bias))
    pred[i] = a * test_blend[i] + b

# tail shrink + bounds
hi = np.quantile(y.values, 0.97)
pred = np.where(pred > hi, hi + 0.987*(pred-hi), pred)
pred = np.clip(pred, 25.0, 500.0)

sub = pd.DataFrame({'id': test_ids, 'aqi': pred}).sort_values('id').reset_index(drop=True)
os.makedirs('working', exist_ok=True)
sub.to_csv('working/submission.csv', index=False)

print('Saved: working/submission.csv')
print('Shape:', sub.shape)
print(sub.head())
